## Respuesta a consigna #5


Sean $A = \begin{bmatrix} a & b \\ c & d \end{bmatrix}$, $B = \begin{bmatrix} x & y \\ z & w \end{bmatrix}$ y por lo tanto $B^T = \begin{bmatrix} x & z \\ y & w \end{bmatrix}$.

$$A \times B^T = \begin{bmatrix} (ax + by) & (az + bw) \\ (cx + dy) & (cz + dw) \end{bmatrix}$$

$$diag(A \times B^T) = \begin{bmatrix} ax + by \\ cz + dw \end{bmatrix}$$

Por otro lado, el producto punto entre A y B es:

$$A \cdot B = \begin{bmatrix} ax & by \\ cz & dw \end{bmatrix}$$

y sumando sobre las columnas:

$$\sum_{cols} (A \cdot B) = \begin{bmatrix} ax + by \\ cz + dw \end{bmatrix}$$

Ambos resultados coinciden, por lo tanto $diag(A \times B^T) = \sum_{cols} (A \cdot B)$.

## Respuesta a consigna #8


Sea $A$ una matriz definida positiva con factorización de Cholesky $A = LL^T$, donde $L$ es triangular inferior (e invertible).

Invirtiendo ambos lados y usando que $(XY)^{-1} = Y^{-1}X^{-1}$:

$$A^{-1} = (LL^T)^{-1} = (L^T)^{-1} L^{-1}$$

Como inversa y transpuesta conmutan, $(L^T)^{-1} = (L^{-1})^T$, y por lo tanto:

$$A^{-1} = (L^{-1})^T L^{-1}$$

**Utilidad en la forma cuadrática de QDA.** Para cada clase hay que calcular $(x-\mu)^T \Sigma^{-1} (x-\mu)$. Tomando $\Sigma = LL^T$ y reemplazando:

$$(x-\mu)^T \Sigma^{-1} (x-\mu) = (x-\mu)^T (L^{-1})^T L^{-1} (x-\mu) = \big(L^{-1}(x-\mu)\big)^T \big(L^{-1}(x-\mu)\big) = \|L^{-1}(x-\mu)\|^2$$

Sea $y = L^{-1}(x-\mu)$. Calcular $y$ equivale a resolver el sistema $Ly = x-\mu$, que al ser triangular se resuelve por sustitución hacia adelante en $O(p^2)$, sin necesidad de invertir ninguna matriz. Luego la forma cuadrática es simplemente $\|y\|^2 = \sum_i y_i^2$.

Además, el determinante que aparece en QDA también sale de $L$:

$$\log|\Sigma| = \log|L|^2 = 2\sum_{i=1}^{p} \log L_{ii}$$

Por lo tanto, con una sola factorización de Cholesky por clase se obtienen la forma cuadrática y el log-determinante, evitando calcular $\Sigma^{-1}$.


**Cómo podemos utilizar esto?** Fijemosnos en la función `_predict_log_conditional` en [base/cholesky.py](base/cholesky.py):

```python
def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()
```

- `unbiased_x` es $x-\mu$ y `y = L_inv @ unbiased_x` es $y = L^{-1}(x-\mu)$.
- `(y**2).sum()` es $\|y\|^2$, la forma cuadrática, multiplicada por $-\frac{1}{2}$ como en la log-densidad gaussiana.
- `np.log(L_inv.diagonal().prod())` es $\sum_i \log (L^{-1})_{ii} = -\sum_i \log L_{ii} = -\frac{1}{2}\log|\Sigma|$, que es el término del determinante.

En resumen, `_predict_log_conditional` calcula:

$$\log P(x \mid G=k) = \log|L_k^{-1}| - \tfrac{1}{2}\|L_k^{-1}(x-\mu_k)\|^2$$

(sin la constante $-\frac{p}{2}\log 2\pi$, que es igual para todas las clases).


## Respuesta a consigna #9

**Diferencias entre `QDA_Chol1`y `QDA`**

- Primero revisamos la diferencia en la función `_fit_params`:
    - QDA: `self.inv_covs = [LA.inv(np.cov(...)]` => guarda la inversa de la matriz de covarianzas
    - QDA_Chol1: `self.L_invs = [LA.inv(cholesky(np.cov(...)))]` => calcula la decomposicion de Cholesky de la matriz de covarianza y guarda su inversa.
- Luego analizemos `_predict_log_conditional`:
    - QDA: la prediccion se calcula de la siguiente manera => `0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x`
    - QDA_Chol1: la preduccion se calcula como `np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()` donde `y = L_inv @ unbiased_x`
    - La igualdad matematica de las dos expresiones quedo demostrada en la respuesta de la consigna #8. Lo que podemos observar es que en la forma de Choleski se simplifica el calculo de la determinante ya que es una matriz triangular.

- **QDA_Chol1** paso a paso:
    1. En `fit`, para cada clase se calcula la covarianza $\Sigma$ y su factorización de Cholesky $\Sigma = LL^T$. Se guarda $L^{-1}$ en `L_invs` (en vez de $\Sigma^{-1}$) y la media de la clase en `means`.
    2. En `predict`, para cada observación $x$ y cada clase se calcula el dato: `unbiased_x = x - mean`, o sea $x-\mu$.
    3. Se multiplica el dato por $L^{-1}$: `y = L_inv @ unbiased_x`, o sea $y = L^{-1}(x-\mu)$.
    4. Se calcula la forma cuadrática como `(y**2).sum()`.
    5. Se calcula el término del determinante como `np.log(L_inv.diagonal().prod())`.
    6. Se combinan ambos términos, $\log|L^{-1}| - \frac{1}{2}\|y\|^2$, que es igual al valor que devuelve `QDA`. Luego, como en `QDA`, se suma `log_a_priori` y se elige la clase con mayor valor (`argmax`).
